# Speech transcription using Whipser model

This project demonstrates an end‑to‑end audio processing pipeline that converts speech to text using OpenAI Whisper and then generates concise summaries using BART (facebook/bart-large-cnn). A Gradio web interface is provided for easy interaction

Features

* Automatic Speech Recognition (ASR) with Whisper

* Multi‑language support: auto, en, fr, ar, es

* Text summarization with BART‑Large‑CNN

* Interactive Gradio UI (microphone + audio upload)



## Set  up and imports

In [1]:
!pip install torch transformers
!pip install openai-whisper
!pip install gradio
!pip install soundfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 10.3 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=5bcd9f4d898db487e3657b2da67d571dc9b0e22da5e25181304d9dbc5ee48112
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 1.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 2.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 6.4 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.5
    Uninstalling pydantic_core-2.41.5:
      Successfully uninstalled pydantic_core-2.41.5
  Attempting un

In [2]:
import whisper
import torch
from transformers import pipeline
import warnings
import gradio as gr
warnings.filterwarnings('ignore')

2025-12-25 14:57:38.733818: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766674658.995838      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766674659.075505      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766674659.756814      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766674659.756874      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766674659.756877      55 computation_placer.cc:177] computation placer alr

## Whisper model loading

In [3]:
def load_whisper_model(model_size="base"):
    """Load Whisper model for transcription"""
    print(f"Loading Whisper {model_size} model...")
    model = whisper.load_model(model_size)
    print(f" Whisper model loaded successfully")
    return model

In [4]:
whisper_model = load_whisper_model("base")

Loading Whisper base model...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 102MiB/s]


 Whisper model loaded successfully


## Summarization model loading

In [5]:
print("Loading summarization model...")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
print("Summarizer loaded")

Loading summarization model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


Summarizer loaded


## Transcription Function

In [6]:
def transcribe_audio(audio_path, language="en"):
    """
    Transcribe audio file to text
    
    Args:
        audio_path: path to audio file
        language: language code (en, ar, fr, es, etc.) or "auto"
    
    Returns:
        transcribed text
    """
    print(f"🎤 Transcribing: {audio_path}")
    
    result = whisper_model.transcribe(
        audio_path,
        language=None if language == "auto" else language
    )
    
    text = result["text"]
    print(f"✅ Transcription complete: {len(text)} characters")
    return text


## Summarization function

In [7]:
def summarize_text(text, max_length=130, min_length=30):
    """
    Summarize text using AI
    
    Args:
        text: text to summarize
        max_length: maximum summary length
        min_length: minimum summary length
    
    Returns:
        summary text
    """
    print("Generating summary...")
    
    max_chunk = 1024
    if len(text) > max_chunk:
        
        text = text[:max_chunk]
    
    summary = summarizer(text, max_length=max_length, min_length=min_length, do_sample=False)
    summary_text = summary[0]['summary_text']
    
    print(f"Summary complete: {len(summary_text)} characters")
    return summary_text

## Complete Pipeline

In [8]:
def transcribe_and_summarize(audio_path, language="en"):
    """
    Complete pipeline: Audio → Transcription → Summary
    
    Args:
        audio_path: path to audio file
        language: language code
    
    Returns:
        dict with transcription and summary
    """
    print("="*60)
    print("🎯 AUDIO TRANSCRIPTION & SUMMARIZATION")
    print("="*60)
    
    # Step 1: Transcribe
    transcription = transcribe_audio(audio_path, language)
    
    # Step 2: Summarize
    summary = summarize_text(transcription)
    
    # Results
    results = {
        "audio_file": audio_path,
        "transcription": transcription,
        "summary": summary,
        "word_count": len(transcription.split())
    }
    
    print("\n" + "="*60)
    print("📄 TRANSCRIPTION:")
    print("="*60)
    print(transcription)
    print("\n" + "="*60)
    print("📋 SUMMARY:")
    print("="*60)
    print(summary)
    print("="*60)
    
    return results

# Gradio Interface

## Backend Function

In [11]:
def gradio_pipeline(audio, language):
    """
    Gradio wrapper for transcription + summarization
    """
    if audio is None:
        return "No audio provided.", ""

    transcription = whisper_model.transcribe(
        audio,
        language=None if language == "auto" else language
    )["text"]

    max_chunk = 1024
    text = transcription[:max_chunk]

    summary = summarizer(
        text,
        max_length=130,
        min_length=30,
        do_sample=False
    )[0]["summary_text"]

    return transcription, summary


## Gardio UI definition

In [12]:
interface = gr.Interface(
    fn=gradio_pipeline,
    inputs=[
        gr.Audio(
            sources=["microphone", "upload"],
            type="filepath",
            label="🎙️ Record or Upload Audio"
        ),
        gr.Dropdown(
            choices=["auto", "en", "fr", "ar", "es"],
            value="auto",
            label="🌍 Language"
        )
    ],
    outputs=[
        gr.Textbox(label="📄 Transcription", lines=10),
        gr.Textbox(label="📋 Summary", lines=5)
    ],
    title="Speech Transcription & Summarization",
    description="Whisper (ASR) + BART (Summarization) — Kaggle Compatible"
)


## Launch application

In [13]:
interface.launch(
    share=True,
    debug=True,
    inline=True
)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://0df0511537e50e16df.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Your max_length is set to 130, but your input_length is only 88. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=44)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://0df0511537e50e16df.gradio.live
